# Load CARWatch logs and Study Manager results

This notebook demonstrates the package capabilities available after the raw-log and Study Manager loaders. It uses synthetic data from `examples/data` and does not access the ignored `playground` directory.

Register the project environment before opening the notebook with `uv run poe conf_jupyter`, then select the `carwatch` kernel.

In [ ]:
from pathlib import Path

import carwatch as cw

In [ ]:
DATA_DIR = Path("examples/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATA_DIR

## Raw app logs

`load_logs` parses semicolon-delimited app logs, converts Unix milliseconds into timezone-aware timestamps, and retains JSON payloads as dictionaries.

In [ ]:
logs = cw.io.load_logs(
    DATA_DIR / "carwatch_demo_VP01_20250515.csv",
    tz="Europe/Berlin",
)
logs[["timestamp", "action", "source_file"]]

## Extract sampling and awakening events

The extraction functions retain the expected sample, the physical tube that was scanned, and any mismatch between them.

In [ ]:
samples = cw.logs.extract_samples(logs)
awakening = cw.logs.extract_awakening(logs)

samples[[
    "sampling_time",
    "sample",
    "sample_scanned",
    "barcode",
    "sample_mismatch",
]]

In [ ]:
awakening

## Study Manager export

`load_study_results` reshapes the wide export into one row per expected sample and calculates sampling time in minutes relative to awakening.

In [ ]:
study_results = cw.io.load_study_results(
    DATA_DIR / "study_results.csv",
    tz="Europe/Berlin",
)
study_results[[
    "sampling_time",
    "time",
    "barcode",
    "sample_scanned",
    "sample_mismatch",
]]

## Audit recorded sample swaps

Filtering `sample_mismatch` exposes the physical tubes that were accidentally exchanged. Correction is handled by a later package capability.

In [ ]:
mismatches = study_results.loc[study_results["sample_mismatch"]]
mismatches[["barcode", "sample_scanned", "mismatch_summary"]]